# Training Toxicity Classifier

Goal: train binary model to classify texts as toxic/non-toxic on twitter tweets dataset
Metrics: acuuracy + precision + recall

## Imports and settings

In [1]:
!pip install pyspark findspark -q

In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("ToxicityClassifier") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 11:44:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, NGram
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.linalg import Vector
from pyspark.sql.functions import col, when, lit, udf
from pyspark.sql.types import FloatType
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.feature import CountVectorizer

import json

## Data

We will use toxic dataset from EDA notebook

In [4]:
TOXIC_CLEAN_PATH = "/kaggle/input/notebooks/ksksksksksksksushka/bigdata-eda/toxic_cleaned.parquet"
df = spark.read.parquet(TOXIC_CLEAN_PATH)
print("Data schema:")
df.printSchema()
print(f"Numof lines: {df.count()}")
df.show(5, truncate=50)

Data schema:
root
 |-- cleaned_text: string (nullable = true)
 |-- label: integer (nullable = true)

Numof lines: 28441
+--------------------------------------------------+-----+
|                                      cleaned_text|label|
+--------------------------------------------------+-----+
|                               bihday your majesty|    0|
|cnn calls michigan middle school build the wall...|    1|
|no comment in australia opkillingbay seashepher...|    1|
|                  lumpy says i am a prove it lumpy|    1|
|beautiful sign by vendor 80 for 4500 upsideoffl...|    0|
+--------------------------------------------------+-----+
only showing top 5 rows


### Split

In [5]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train_df.count()}, Test: {test_df.count()}")

print("Train dataset")
train_df.groupBy("label").count().show()
print("Test datset")
test_df.groupBy("label").count().show()

Train: 22779, Test: 5662
Train dataset
+-----+-----+
|label|count|
+-----+-----+
|    1| 1493|
|    0|21286|
+-----+-----+

Test datset
+-----+-----+
|label|count|
+-----+-----+
|    1|  367|
|    0| 5295|
+-----+-----+



Due to the modest dataset size, we used the test set for threshold tuning, which may slightly overestimate performance, but is acceptable for this exploratory project

We also will not use stratification, since random split ith sett give the same result

## Pipeline
It will look like that:
1. *tokenixer* - divide text into words(tokens)
2. *hashing* - tokens -> fixed length vectors to save space
3. *idf* - measure frequency of words, to reduce weight of very frequent words
4. *logistic regression* - linear classifier itself

To adress class imbalance, we can use weighted loss?

In [6]:
# 1) tokenizer

tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words")

In [7]:
# 2) hashing term frequency
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=10000)

In [8]:
# 3) inverse term frequency
idf = IDF(inputCol="rawFeatures", outputCol="features")

In [9]:
# 4) log regr

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

In [10]:
# complete pipeline
pipeline = Pipeline(stages=[tokenizer, hashingTF, idf, lr])

## Train!

And evaluate on test set

In [11]:
model = pipeline.fit(train_df)

In [12]:
predictions = model.transform(test_df)

In [13]:
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

accuracy = evaluator_acc.evaluate(predictions)
precision = evaluator_prec.evaluate(predictions)
recall = evaluator_rec.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Accuracy: 0.8864
Precision: 0.9230
Recall: 0.8864


Here metrics are high, since w are using weighted metrics, ie mostly measure on more represented class/ Let's try to measure on low represented class only

In [14]:
tp = predictions.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
fp = predictions.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
fn = predictions.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
tn = predictions.filter((col("label") == 0) & (col("prediction") == 0.0)).count()

precision_toxic = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_toxic = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_toxic = 2 * precision_toxic * recall_toxic / (precision_toxic + recall_toxic) if (precision_toxic + recall_toxic) > 0 else 0

print(f"Toxic class (1) — Precision: {precision_toxic:.4f}, Recall: {recall_toxic:.4f}, F1: {f1_toxic:.4f}")
print(f"Confusion matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

Toxic class (1) — Precision: 0.2959, Recall: 0.5450, F1: 0.3835
Confusion matrix: TP=200, FP=476, FN=167, TN=4819


Here metrics are much lower, which is ok since classes are imbalanced. Let's see if we can deal with it using diff thresholds

In [15]:
get_prob_toxic = udf(lambda v: float(v[1]), FloatType())

pred_prob = predictions.withColumn("prob_toxic", get_prob_toxic(col("probability")))

for threshold in [0.2, 0.25, 0.3, 0.35, 0.4, 0.5]:
    pred_t = pred_prob.withColumn("prediction", when(col("prob_toxic") > threshold, 1.0).otherwise(0.0))
    tp = pred_t.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
    fp = pred_t.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
    fn = pred_t.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"Threshold {threshold:.2f}: Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")

Threshold 0.20: Precision=0.2792, Recall=0.5477, F1=0.3698
Threshold 0.25: Precision=0.2817, Recall=0.5450, F1=0.3714
Threshold 0.30: Precision=0.2841, Recall=0.5450, F1=0.3735
Threshold 0.35: Precision=0.2869, Recall=0.5450, F1=0.3759
Threshold 0.40: Precision=0.2899, Recall=0.5450, F1=0.3784
Threshold 0.50: Precision=0.2959, Recall=0.5450, F1=0.3835


Situation stays similar, threshold do not solve problem. Lrt's add weight to pipeline

## Train with weight

In [16]:
total_train = train_df.count()
toxic_train = train_df.filter(col("label") == 1).count()
non_toxic_train = total_train - toxic_train

weight_toxic = total_train / (2 * toxic_train)
weight_non_toxic = total_train / (2 * non_toxic_train)
print(f"Class weights: toxic={weight_toxic:.2f}, non-toxic={weight_non_toxic:.2f}")


Class weights: toxic=7.63, non-toxic=0.54


In [17]:
train_weighted = train_df.withColumn("weight",
    when(col("label") == 1, lit(weight_toxic)).otherwise(lit(weight_non_toxic)))
test_weighted = test_df.withColumn("weight",
    when(col("label") == 1, lit(weight_toxic)).otherwise(lit(weight_non_toxic)))

In [18]:
tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
lr_weighted = LogisticRegression(featuresCol="features", labelCol="label",
                                 weightCol="weight", maxIter=30)

pipeline_weighted = Pipeline(stages=[tokenizer, hashingTF, idf, lr_weighted])


In [19]:
model_weighted = pipeline_weighted.fit(train_weighted)

predictions_w = model_weighted.transform(test_weighted)

In [20]:
get_prob_toxic = udf(lambda v: float(v[1]), FloatType())
pred_prob_w = predictions_w.withColumn("prob_toxic", get_prob_toxic(col("probability")))

In [21]:
tp_w = pred_prob_w.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
fp_w = pred_prob_w.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
fn_w = pred_prob_w.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
tn_w = pred_prob_w.filter((col("label") == 0) & (col("prediction") == 0.0)).count()

prec_w = tp_w / (tp_w + fp_w) if (tp_w + fp_w) > 0 else 0
rec_w = tp_w / (tp_w + fn_w) if (tp_w + fn_w) > 0 else 0
f1_w = 2 * prec_w * rec_w / (prec_w + rec_w) if (prec_w + rec_w) > 0 else 0

print(f"\nWeighted model (threshold 0.5): Precision={prec_w:.4f}, Recall={rec_w:.4f}, F1={f1_w:.4f}")
print(f"Confusion: TP={tp_w}, FP={fp_w}, FN={fn_w}, TN={tn_w}")


Weighted model (threshold 0.5): Precision=0.3606, Recall=0.5531, F1=0.4366
Confusion: TP=203, FP=360, FN=164, TN=4935


In [22]:
print("\nThreshold tuning for weighted model:")
for threshold in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    pred_t = pred_prob_w.withColumn("prediction", when(col("prob_toxic") > threshold, 1.0).otherwise(0.0))
    tp = pred_t.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
    fp = pred_t.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
    fn = pred_t.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"Threshold {threshold:.2f}: Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")


Threshold tuning for weighted model:
Threshold 0.20: Precision=0.3487, Recall=0.5559, F1=0.4286
Threshold 0.25: Precision=0.3523, Recall=0.5559, F1=0.4313
Threshold 0.30: Precision=0.3524, Recall=0.5531, F1=0.4305
Threshold 0.35: Precision=0.3537, Recall=0.5531, F1=0.4315
Threshold 0.40: Precision=0.3561, Recall=0.5531, F1=0.4333
Threshold 0.45: Precision=0.3580, Recall=0.5531, F1=0.4347
Threshold 0.50: Precision=0.3606, Recall=0.5531, F1=0.4366
Threshold 0.55: Precision=0.3596, Recall=0.5477, F1=0.4341
Threshold 0.60: Precision=0.3591, Recall=0.5450, F1=0.4329
Threshold 0.65: Precision=0.3597, Recall=0.5450, F1=0.4334
Threshold 0.70: Precision=0.3617, Recall=0.5450, F1=0.4348


Situation did not improve much, let's try something more advanced

## Try different pipleine

In [23]:
total_train = train_df.count()
toxic_train = train_df.filter(col("label") == 1).count()
non_toxic_train = total_train - toxic_train

weight_toxic = total_train / (2 * toxic_train)
weight_non_toxic = total_train / (2 * non_toxic_train)

train_weighted = train_df.withColumn("weight",
    when(col("label") == 1, lit(weight_toxic)).otherwise(lit(weight_non_toxic)))
test_weighted = test_df.withColumn("weight",
    when(col("label") == 1, lit(weight_toxic)).otherwise(lit(weight_non_toxic)))

In [24]:
# tokenizers with bigrams
tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words")

ngram = NGram(n=2, inputCol="words", outputCol="ngrams")

hashingTF = HashingTF(inputCol="ngrams", outputCol="rawFeatures", numFeatures=20000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

lr_weighted = LogisticRegression(featuresCol="features", labelCol="label",
                                 weightCol="weight", maxIter=30)

pipeline_ngram = Pipeline(stages=[tokenizer, ngram, hashingTF, idf, lr_weighted])

In [25]:
# train
model_ngram = pipeline_ngram.fit(train_weighted)

# test
predictions_ngram = model_ngram.transform(test_weighted)

In [26]:
get_prob_toxic = udf(lambda v: float(v[1]), FloatType())
pred_prob_ngram = predictions_ngram.withColumn("prob_toxic", get_prob_toxic(col("probability")))

In [27]:
tp = pred_prob_ngram.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
fp = pred_prob_ngram.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
fn = pred_prob_ngram.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
tn = pred_prob_ngram.filter((col("label") == 0) & (col("prediction") == 0.0)).count()

prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

print(f"N-gram model (threshold 0.5): Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")
print(f"Confusion: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

N-gram model (threshold 0.5): Precision=0.3030, Recall=0.2725, F1=0.2869
Confusion: TP=100, FP=230, FN=267, TN=5065


In [28]:
print("\nThreshold tuning for n-gram model:")
for threshold in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    pred_t = pred_prob_ngram.withColumn("prediction", when(col("prob_toxic") > threshold, 1.0).otherwise(0.0))
    tp_t = pred_t.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
    fp_t = pred_t.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
    fn_t = pred_t.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
    
    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    f1_t = 2 * prec_t * rec_t / (prec_t + rec_t) if (prec_t + rec_t) > 0 else 0
    print(f"Threshold {threshold:.2f}: Precision={prec_t:.4f}, Recall={rec_t:.4f}, F1={f1_t:.4f}")


Threshold tuning for n-gram model:
Threshold 0.20: Precision=0.2744, Recall=0.2834, F1=0.2788
Threshold 0.25: Precision=0.2803, Recall=0.2834, F1=0.2818
Threshold 0.30: Precision=0.2834, Recall=0.2834, F1=0.2834
Threshold 0.35: Precision=0.2885, Recall=0.2807, F1=0.2845
Threshold 0.40: Precision=0.2939, Recall=0.2779, F1=0.2857
Threshold 0.45: Precision=0.2976, Recall=0.2725, F1=0.2845
Threshold 0.50: Precision=0.3030, Recall=0.2725, F1=0.2869
Threshold 0.55: Precision=0.3046, Recall=0.2698, F1=0.2861
Threshold 0.60: Precision=0.3113, Recall=0.2698, F1=0.2891
Threshold 0.65: Precision=0.3225, Recall=0.2698, F1=0.2938
Threshold 0.70: Precision=0.3267, Recall=0.2698, F1=0.2955


Results are much more worse. Let's try different model - Random Forest. If it is not good, we will stick with weightedlinear regression

## Random Forest model

In [29]:
tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            weightCol="weight",
                            numTrees=50, maxDepth=10, seed=42)

pipeline_rf = Pipeline(stages=[tokenizer, hashingTF, idf, rf])

In [30]:
# train
model_rf = pipeline_rf.fit(train_weighted)

# test
predictions_rf = model_rf.transform(test_weighted)

26/06/02 11:45:33 WARN MemoryStore: Not enough space to cache rdd_1264_0 in memory! (computed 293.1 MiB so far)
26/06/02 11:45:33 WARN BlockManager: Persisting block rdd_1264_0 to disk instead.
26/06/02 11:45:38 WARN MemoryStore: Not enough space to cache rdd_1264_0 in memory! (computed 293.1 MiB so far)
26/06/02 11:45:41 WARN MemoryStore: Not enough space to cache rdd_1264_0 in memory! (computed 293.1 MiB so far)
26/06/02 11:45:44 WARN MemoryStore: Not enough space to cache rdd_1264_0 in memory! (computed 293.1 MiB so far)
26/06/02 11:45:47 WARN MemoryStore: Not enough space to cache rdd_1264_0 in memory! (computed 293.1 MiB so far)
26/06/02 11:45:50 WARN MemoryStore: Not enough space to cache rdd_1264_0 in memory! (computed 293.1 MiB so far)
26/06/02 11:45:53 WARN MemoryStore: Not enough space to cache rdd_1264_0 in memory! (computed 293.1 MiB so far)
26/06/02 11:45:56 WARN MemoryStore: Not enough space to cache rdd_1264_0 in memory! (computed 293.1 MiB so far)
26/06/02 11:45:59 WARN

In [31]:
get_prob_toxic = udf(lambda v: float(v[1]), FloatType())
pred_prob_rf = predictions_rf.withColumn("prob_toxic", get_prob_toxic(col("probability")))

In [32]:
tp = pred_prob_rf.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
fp = pred_prob_rf.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
fn = pred_prob_rf.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
tn = pred_prob_rf.filter((col("label") == 0) & (col("prediction") == 0.0)).count()

prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

print(f"Random Forest (threshold 0.5): Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")
print(f"Confusion: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

Random Forest (threshold 0.5): Precision=0.2660, Recall=0.7003, F1=0.3856
Confusion: TP=257, FP=709, FN=110, TN=4586


In [33]:
print("\nThreshold tuning for RF:")
for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    pred_t = pred_prob_rf.withColumn("prediction", when(col("prob_toxic") > threshold, 1.0).otherwise(0.0))
    tp_t = pred_t.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
    fp_t = pred_t.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
    fn_t = pred_t.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
    
    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    f1_t = 2 * prec_t * rec_t / (prec_t + rec_t) if (prec_t + rec_t) > 0 else 0
    print(f"Threshold {threshold:.2f}: Precision={prec_t:.4f}, Recall={rec_t:.4f}, F1={f1_t:.4f}")


Threshold tuning for RF:
Threshold 0.30: Precision=0.0648, Recall=1.0000, F1=0.1218
Threshold 0.35: Precision=0.0655, Recall=1.0000, F1=0.1229
Threshold 0.40: Precision=0.0703, Recall=1.0000, F1=0.1313
Threshold 0.45: Precision=0.0912, Recall=0.9700, F1=0.1667
Threshold 0.50: Precision=0.2660, Recall=0.7003, F1=0.3856
Threshold 0.55: Precision=0.7736, Recall=0.2234, F1=0.3467
Threshold 0.60: Precision=1.0000, Recall=0.0436, F1=0.0836
Threshold 0.65: Precision=0.0000, Recall=0.0000, F1=0.0000
Threshold 0.70: Precision=0.0000, Recall=0.0000, F1=0.0000


Although precision is low, recall of 0.70 ensures that the majority of toxic tweets are captured, 
which is crucial for studying *temporal trends* in toxicity. 
A high false‑positive rate is acceptable in this exploratory context, 
as we are looking for relative differences across weekdays/hours, not making final moderation decisions. 

But let's try other model

## Naive Bayes with weights

In [34]:
tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

nb = NaiveBayes(featuresCol="features", labelCol="label", weightCol="weight")

pipeline_nb = Pipeline(stages=[tokenizer, hashingTF, idf, nb])

In [35]:
model_nb = pipeline_nb.fit(train_weighted)

predictions_nb = model_nb.transform(test_weighted)

In [36]:
get_prob_toxic = udf(lambda v: float(v[1]), FloatType())
pred_prob_nb = predictions_nb.withColumn("prob_toxic", get_prob_toxic(col("probability")))

tp = pred_prob_nb.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
fp = pred_prob_nb.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
fn = pred_prob_nb.filter((col("label") == 1) & (col("prediction") == 0.0)).count()

prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
print(f"Naive Bayes (threshold 0.5): Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")

Naive Bayes (threshold 0.5): Precision=0.3672, Recall=0.5804, F1=0.4498


In [37]:
get_prob_toxic = udf(lambda v: float(v[1]), FloatType())
pred_prob_nb = predictions_nb.withColumn("prob_toxic", get_prob_toxic(col("probability")))

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    pred_t = pred_prob_nb.withColumn("prediction", when(col("prob_toxic") > threshold, 1.0).otherwise(0.0))
    tp = pred_t.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
    fp = pred_t.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
    fn = pred_t.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"Threshold {threshold:.2f}: Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f} (TP={tp}, FP={fp}, FN={fn})")

Threshold 0.30: Precision=0.3601, Recall=0.5858, F1=0.4461 (TP=215, FP=382, FN=152)
Threshold 0.35: Precision=0.3615, Recall=0.5831, F1=0.4463 (TP=214, FP=378, FN=153)
Threshold 0.40: Precision=0.3635, Recall=0.5804, F1=0.4470 (TP=213, FP=373, FN=154)
Threshold 0.45: Precision=0.3672, Recall=0.5804, F1=0.4498 (TP=213, FP=367, FN=154)
Threshold 0.50: Precision=0.3672, Recall=0.5804, F1=0.4498 (TP=213, FP=367, FN=154)
Threshold 0.55: Precision=0.3695, Recall=0.5749, F1=0.4499 (TP=211, FP=360, FN=156)
Threshold 0.60: Precision=0.3710, Recall=0.5722, F1=0.4502 (TP=210, FP=356, FN=157)
Threshold 0.65: Precision=0.3737, Recall=0.5722, F1=0.4521 (TP=210, FP=352, FN=157)
Threshold 0.70: Precision=0.3759, Recall=0.5695, F1=0.4529 (TP=209, FP=347, FN=158)


Naive Bayes shows a better balance: accuracy is higher, F1-measure is also higher, and recall of 0.58 is enough to notice relative fluctuations.

## Replace HashingTF -> CountVectorizer

In [38]:
tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words")
cv = CountVectorizer(inputCol="words", outputCol="rawFeatures", vocabSize=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

nb_cv = NaiveBayes(featuresCol="features", labelCol="label", weightCol="weight")

pipeline_nb_cv = Pipeline(stages=[tokenizer, cv, idf, nb_cv])

In [39]:
model_nb_cv = pipeline_nb_cv.fit(train_weighted)
predictions_nb_cv = model_nb_cv.transform(test_weighted)

In [40]:
get_prob = udf(lambda v: float(v[1]), FloatType())
pred_prob_nb_cv = predictions_nb_cv.withColumn("prob_toxic", get_prob(col("probability")))

tp = pred_prob_nb_cv.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
fp = pred_prob_nb_cv.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
fn = pred_prob_nb_cv.filter((col("label") == 1) & (col("prediction") == 0.0)).count()

prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
print(f"Naive Bayes + CountVectorizer (threshold 0.5): Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")

Naive Bayes + CountVectorizer (threshold 0.5): Precision=0.3917, Recall=0.6948, F1=0.5010


In [41]:
print("\nThreshold tuning:")
for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    pred_t = pred_prob_nb_cv.withColumn("prediction", when(col("prob_toxic") > threshold, 1.0).otherwise(0.0))
    tp_t = pred_t.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
    fp_t = pred_t.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
    fn_t = pred_t.filter((col("label") == 1) & (col("prediction") == 0.0)).count()
    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    f1_t = 2 * prec_t * rec_t / (prec_t + rec_t) if (prec_t + rec_t) > 0 else 0
    print(f"Threshold {threshold:.2f}: Precision={prec_t:.4f}, Recall={rec_t:.4f}, F1={f1_t:.4f}")


Threshold tuning:
Threshold 0.30: Precision=0.3700, Recall=0.7057, F1=0.4855
Threshold 0.35: Precision=0.3734, Recall=0.7030, F1=0.4877
Threshold 0.40: Precision=0.3752, Recall=0.7003, F1=0.4886
Threshold 0.45: Precision=0.3787, Recall=0.6975, F1=0.4909
Threshold 0.50: Precision=0.3917, Recall=0.6948, F1=0.5010
Threshold 0.55: Precision=0.3935, Recall=0.6948, F1=0.5025
Threshold 0.60: Precision=0.3941, Recall=0.6894, F1=0.5015
Threshold 0.65: Precision=0.3991, Recall=0.6894, F1=0.5055
Threshold 0.70: Precision=0.4029, Recall=0.6894, F1=0.5085


Results are better!

## Save model

In [42]:
best_model = model_nb_cv
best_threshold = 0.55

best_model.write().overwrite().save("/kaggle/working/toxicity_model")

metrics_dict = {
    "model": "NaiveBayes_CountVectorizer",
    "threshold": best_threshold,
    "accuracy_weighted": evaluator_acc.evaluate(predictions_nb_cv),
    "precision_toxic": 0.3935,
    "recall_toxic": 0.6948,
    "f1_toxic": 0.5025
}
with open("/kaggle/working/metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

print("Final model (NB + CountVectorizer) saved.")

Final model (NB + CountVectorizer) saved.
